# 轮次

在深度学习中，无论采用哪种梯度下降的方法，将整个训练集遍历一遍的过程称为一个**轮次**（Epoch）。

如果训练集有 $n$ 个样本：
* **随机梯度下降**：每轮进行 $n$ 次迭代。
* **批量梯度下降**：每轮仅 1 次迭代。
* **小批量梯度下降**（批大小 $m$）：每轮进行 $n/m$ 次迭代。

通常，网络模型需要**多个轮次**的重复训练，才能使参数充分收敛到较优状态。

In [14]:
import numpy as np

## 张量

In [15]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    def __str__(self):
        return f'Tensor({self.data})'

## 数据集

In [16]:
class Dataset:

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.load()
        self.train()

    def load(self):
        self.train_data = ([[22.5, 72.0],
                            [31.4, 45.0],
                            [19.8, 85.0],
                            [27.6, 63.0]],
                           [[95],
                            [210],
                            [70],
                            [155]])
        self.test_data = ([[28.1, 58.0]],
                          [[165]])

    def train(self):
        self.data = self.train_data

    def eval(self):
        self.data = self.test_data

    def all(self):
        x, y = self.data
        return Tensor(x), Tensor(y)

    def __len__(self):
        x, *_ = self.data
        return len(x) // self.batch_size

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y = self.data
        return Tensor(x[s]), Tensor(y[s])

## 模型

In [17]:
class Linear:

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def __call__(self, x: Tensor):
        return self.forward(x)

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)

        p.gradient_fn = gradient_fn
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

## 损失函数（均方误差）

In [18]:
class MSELoss:

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / y.data.size

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

## 优化器（随机梯度下降）

In [19]:
class SGDOptimizer:

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

## 训练器（神经元网络）

In [20]:
class NNTrainer:

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    def train(self, dataset, epochs):
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction = self.layer(feature)
                loss = self.loss_fn(prediction, label)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        dataset.eval()

        feature, label = dataset.all()
        prediction = self.layer(feature)
        loss = self.loss_fn(prediction, label)
        return prediction, loss

## 超参数

### 学习率

In [21]:
LEARNING_RATE = 0.00001

### 批大小

In [22]:
BATCH_SIZE = 2

### 轮数

这是我们的第三个超参数。**轮数**（Epochs）定义了模型训练中，训练数据的重复迭代次数。

轮数并非越多越好：

* 在训练初期，模型参数通常会随着轮次增加而快速收敛，误差大幅下降；
* 随后，收敛速度逐渐趋于平缓；
* 如果轮数过多，模型可能会由于**过拟合**，导致在实际应用中的效果反而变差。

因此，我们需要通过监测网络模型在测试数据上的表现，找到性能提升的**拐点**（Turning Point），从而确定最佳轮数：

* 轮数太少：欠拟合。
* 轮数太多：可能过拟合。

一个有效的实践技巧叫做**早停**（Early Stopping）。就是在每个轮次结束后，和上一轮对比。如果损失量没有明显降低（比如：大于 1%），就停止训练。

In [23]:
EPOCHS = 1000

## 建模

In [24]:
dataset = Dataset(BATCH_SIZE)
layer = Linear(2, 1)
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)
trainer = NNTrainer(layer, loss_fn, optimizer)

## 训练

In [25]:
trainer.train(dataset, EPOCHS)

## 评估

In [26]:
prediction, loss = trainer.test(dataset)
print(f'prediction:\t{prediction}\nloss:\t{loss}')

prediction:	Tensor([[163.52327795]])
loss:	Tensor(2.1807080003283335)
